# Kintsugi exploratory demonstration

Marketing records are augmented with simulated finances. Scores identify anomalies, not default probabilities. Historical outputs have been cleared. See models/isolation_forest.metadata.json for current provenance and held-out comparisons.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path for robust module resolution
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

from backend.config import BASE_DIR, RAW_DATA_DIR, FEATURE_COLUMNS, TIER_1_LOW_RISK_MAX, TIER_2_MODERATE_STRESS_MAX
from backend.data.loader import load_customer_data
from backend.data.feature_engineering import compute_stress_features, get_feature_matrix
from backend.agents.detection_agent import StressDetectionAgent

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries and Kintsugi AI modules successfully imported!")
print(f"Target dataset path: {RAW_DATA_DIR / 'marketing_campaign.csv'}")

## 1. Portfolio Ingestion & Schema Validation
We load the official 2,240-row customer dataset from `data/raw/marketing_campaign.csv` using Kintsugi AI's standardized ingestion pipeline `load_customer_data()`.

In [ ]:
# Load official marketing campaign customer dataset (2,240 rows)
raw_csv_path = PROJECT_ROOT / "data" / "raw" / "marketing_campaign.csv"
df_customers = load_customer_data(str(raw_csv_path))

print(f"Successfully loaded {len(df_customers):,} customer profiles.")
print(f"Total attributes: {df_customers.shape[1]}")
df_customers[['customer_id', 'name', 'monthly_income', 'monthly_expenses', 'current_emi', 'deal_purchase_ratio', 'discretionary_ratio']].head()

## 2. Non-Linear Financial Distress Feature Engineering
Raw customer variables are transformed into standardized non-linear distress indicators via `compute_stress_features()`:
- **Spend-to-Income Ratio**: Essential consumption burden relative to monthly income.
- **EMI Burden Ratio**: Debt servicing commitments relative to monthly cash inflows.
- **Deal Reliance Index**: Frequency of coupon/promotional purchases as a share of total purchases.
- **Liquidity Runway (Months)**: Liquid savings buffer expressed in months of essential outflow survival.
- **Savings Depletion Rate**: Velocity at which liquid reserves have dropped over recent quarters.
- **Credit Utilization Ratio**: Credit card balance utilization relative to limits.
- **Discretionary Ratio**: Share of purchases dedicated to luxury/discretionary items (Wines, Sweets, Gold) vs. essential staples.

In [ ]:
# Compute engineered stress features
df_features = compute_stress_features(df_customers)

print(f"Feature matrix dimensions: {df_features.shape}")
print(f"Core Model Features: {FEATURE_COLUMNS}")
df_features[FEATURE_COLUMNS + ['discretionary_ratio', 'stress_index']].describe().T[['mean', 'std', 'min', '50%', 'max']]

## 3. Mathematical Foundations: Unsupervised Isolation Forest & Calibration

### Isolation Forest Theory
Isolation Forest isolates anomalies by randomly selecting a feature and randomly selecting a split value between the maximum and minimum values of that feature. Because anomalies reside in sparse, low-density regions of the feature space, they are isolated in fewer recursive splits than normal observations.

For an ensemble of $t$ isolation trees grown on subsamples of size $n$, the path length $h(x)$ is the number of edges traversed from the root node to the terminating leaf node for borrower $x$.

The average path length of an unsuccessful search in a Binary Search Tree (BST) provides the theoretical normalization baseline:
$$c(n) = 2\left(\ln(n - 1) + \gamma\right) - \frac{2(n - 1)}{n}$$
where $\gamma \approx 0.5772156649$ is the Euler-Mascheroni constant.

The anomaly score $s(x, n)$ for borrower profile $x$ is defined as:
$$s(x, n) = 2^{-\frac{\mathbb{E}(h(x))}{c(n)}}$$

- When $\mathbb{E}(h(x)) \to 0 \implies s \to 1$ (Highly isolated anomaly, acute financial distress).
- When $\mathbb{E}(h(x)) \to c(n) \implies s \to 0.5$ (Typical observation).
- When $\mathbb{E}(h(x)) \to n - 1 \implies s \to 0$ (Deeply clustered normal profile).

### Calibration & Risk Stratification
Kintsugi AI fixes the empirical contamination rate at **15.0%** ($\alpha = 0.15$), matching the upper percentile of high-strain borrowers:
$$\text{Anomalies} = \lfloor N \times \alpha \rfloor = \lfloor 2,240 \times 0.15 \rfloor = 336 \text{ accounts}$$

We calibrate raw decision boundaries into an actionable tripartite credit risk stratification:
- **Tier 1 (Normal / Low Risk)**: $s < 0.35$ $\to$ Routine automated servicing, standard credit monitoring.
- **Tier 2 (Moderate Stress)**: $0.35 \le s < 0.65$ $\to$ Proactive financial wellness alerts, coupon optimization.
- **Tier 3 (High Anomaly / Severe Distress)**: $s \ge 0.65$ $\to$ Early restructuring intervention queue (tenure extension, moratorium).

In [ ]:
# Execute StressDetectionAgent on full customer portfolio
agent = StressDetectionAgent()
df_scored = agent.analyze_portfolio(df_customers)

print("Risk Tier Distribution Summary:")
tier_counts = df_scored['risk_tier'].value_counts()
for tier, count in tier_counts.items():
    pct = (count / len(df_scored)) * 100
    print(f"  • {tier:42s}: {count:5,d} accounts ({pct:5.1f}%)")

print(f"\nTotal Accounts Scored:     {len(df_scored):,}")
print(f"Severe Anomalies Isolated: {df_scored['is_anomaly'].sum():,} ({df_scored['is_anomaly'].mean()*100:.1f}%)")

## 4. Publication-Grade Evaluation Visualizations

Below we display the three publication-grade charts generated by `backend/generate_visualizations.py` and analyze the mathematical and empirical findings.

### Chart A: Continuous Borrower Distress Score Distribution & Anomaly Threshold
The histogram and empirical Kernel Density Estimate (KDE) curve reveal the distribution of calibrated stress scores across the 2,240 borrowers. The vertical dashed line at $s = 0.65$ demarcates the exact **15.0% contamination threshold**, isolating the 336 anomalous accounts (Tier 3) requiring immediate restructuring intervention.

In [ ]:
# Display Chart A: Continuous Distress Distribution
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

chart_a_path = PROJECT_ROOT / "assets" / "distress_distribution.png"
if chart_a_path.exists():
    img_a = mpimg.imread(str(chart_a_path))
    fig, ax = plt.subplots(figsize=(14, 7.5), dpi=150)
    ax.imshow(img_a)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Notice: {chart_a_path} not found. Run python backend/generate_visualizations.py to create it.")

### Descriptive cohort comparison
Cohort differences partly reflect synthetic assumptions. Read current generated charts; no fixed effect size is claimed.


In [ ]:
# Display Chart B: Deal Reliance vs Discretionary Ratio Scatter
chart_b_path = PROJECT_ROOT / "assets" / "deal_vs_spend_scatter.png"
if chart_b_path.exists():
    img_b = mpimg.imread(str(chart_b_path))
    fig, ax = plt.subplots(figsize=(14, 7.5), dpi=150)
    ax.imshow(img_b)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Notice: {chart_b_path} not found. Run python backend/generate_visualizations.py to create it.")

### Descriptive cohort comparison
Cohort differences partly reflect synthetic assumptions. Read current generated charts; no fixed effect size is claimed.


In [ ]:
# Display Chart C: Key Risk Drivers Comparison
chart_c_path = PROJECT_ROOT / "assets" / "risk_drivers_comparison.png"
if chart_c_path.exists():
    img_c = mpimg.imread(str(chart_c_path))
    fig, ax = plt.subplots(figsize=(16, 5.5), dpi=150)
    ax.imshow(img_c)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Notice: {chart_c_path} not found. Run python backend/generate_visualizations.py to create it.")

## 5. Basic pipeline checks
Verify record preservation and finite model inputs. Counts are descriptive and may change after retraining.


In [ ]:
assert len(df_scored) == len(df_customers)
assert not df_scored[FEATURE_COLUMNS].isna().any().any()
print(df_scored["risk_tier"].value_counts())
print("Anomaly indicators only; no observed default validation in this demo.")
